# MathTrans Phase 2 LaTeX

Continue training from the Phase 1 supplementary model on LaTeX-placeholder data scraped from arXiv.

In [ ]:
# Cell 1 - Install dependencies
%pip install -q transformers==4.40.2 peft==0.10.0 accelerate==0.30.1 datasets evaluate sacrebleu sentencepiece matplotlib pandas


In [ ]:
# Cell 2 - Mount Drive / setup paths
from __future__ import annotations

import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from sacrebleu import corpus_bleu
from sacrebleu.metrics import CHRF
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
)

try:
    from google.colab import drive

    drive.mount("/content/drive")
    COLAB_ROOT = Path("/content/drive/MyDrive/MathTrans")
except Exception:
    COLAB_ROOT = Path.cwd()

PROJECT_ROOT = next(
    (path for path in [COLAB_ROOT, COLAB_ROOT / "Math_AI_translation", Path.cwd()] if path.exists()),
    Path.cwd(),
)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "phase2_latex"
CKPT_DIR = OUTPUT_DIR / "checkpoints"
FINAL_DIR = OUTPUT_DIR / "final"
CURVES_PATH = OUTPUT_DIR / "curves.json"

PHASE1_SUPPLEMENTARY_DIR = Path(os.environ.get("PHASE1_SUPPLEMENTARY_DIR", PROJECT_ROOT / "artifacts" / "phase1_clean" / "final"))
DATASET_PATH = DATA_DIR / "phase2_latex_dataset.jsonl"
if not DATASET_PATH.exists():
    DATASET_PATH = DATA_DIR / "phase2_full_v3.jsonl"

MODEL_NAME = "facebook/nllb-200-1.3B"
LORA_CONFIG = dict(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    lora_dropout=0.10,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)
TRAIN_CONFIG = dict(
    max_source_length=256,
    max_target_length=256,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.10,
    num_train_epochs=3,
    weight_decay=0.01,
    label_smoothing_factor=0.0,
    val_split=0.10,
    logging_steps=25,
    eval_steps=100,
    save_steps=100,
    save_total_limit=None,
    eval_num_beams=1,
    eval_generation_max_length=128,
    final_num_beams=4,
    final_generation_max_length=256,
    seed=42,
)

for path in [OUTPUT_DIR, CKPT_DIR, FINAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

random.seed(TRAIN_CONFIG["seed"])
np.random.seed(TRAIN_CONFIG["seed"])
torch.manual_seed(TRAIN_CONFIG["seed"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
chrf_metric = CHRF(word_order=2)
print("Phase 1 supplementary model:", PHASE1_SUPPLEMENTARY_DIR)
print("Phase 2 dataset:", DATASET_PATH)
print("Device:", DEVICE)


In [ ]:
# Cell 3 - Load Phase 1 model and inspect Phase 2 dataset
records = [json.loads(line) for line in DATASET_PATH.open("r", encoding="utf-8")]
random.shuffle(records)
dataset = Dataset.from_list(records)
split_dataset = dataset.train_test_split(
    test_size=TRAIN_CONFIG["val_split"],
    seed=TRAIN_CONFIG["seed"],
)
dataset_dict = DatasetDict(
    train=split_dataset["train"],
    validation=split_dataset["test"],
)
sample_frame = pd.DataFrame(random.sample(records, k=min(3, len(records))))
display(sample_frame)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
phase1_model_ref = str(PHASE1_SUPPLEMENTARY_DIR) if PHASE1_SUPPLEMENTARY_DIR.exists() else MODEL_NAME
phase1_model = AutoModelForSeq2SeqLM.from_pretrained(phase1_model_ref)
model = get_peft_model(phase1_model, LoraConfig(**LORA_CONFIG))
model.print_trainable_parameters()


In [ ]:
# Cell 4 - Tokenize and verify placeholders survive
def preprocess_batch(batch):
    model_inputs = tokenizer(
        batch["source"],
        max_length=TRAIN_CONFIG["max_source_length"],
        truncation=True,
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["target"],
            max_length=TRAIN_CONFIG["max_target_length"],
            truncation=True,
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset_dict.map(
    preprocess_batch,
    batched=True,
    remove_columns=dataset_dict["train"].column_names,
)

placeholder_examples = [row["source"] for row in records if "<MATH_" in row["source"]][:5]
print("Placeholder examples:")
for item in placeholder_examples:
    print(" ", item)


In [ ]:
# Cell 5 - Training loop
class SaveCurvesCallback(TrainerCallback):
    def on_log(self, args, state, control, **kwargs):
        CURVES_PATH.write_text(json.dumps(state.log_history, indent=2), encoding="utf-8")


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    bleu = corpus_bleu(decoded_predictions, [decoded_labels]).score
    chrf = chrf_metric.corpus_score(decoded_predictions, [decoded_labels]).score
    return {"bleu": bleu, "chrf": chrf}


training_args = Seq2SeqTrainingArguments(
    output_dir=str(CKPT_DIR),
    per_device_train_batch_size=TRAIN_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAIN_CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=TRAIN_CONFIG["gradient_accumulation_steps"],
    learning_rate=TRAIN_CONFIG["learning_rate"],
    lr_scheduler_type=TRAIN_CONFIG["lr_scheduler_type"],
    warmup_ratio=TRAIN_CONFIG["warmup_ratio"],
    num_train_epochs=TRAIN_CONFIG["num_train_epochs"],
    weight_decay=TRAIN_CONFIG["weight_decay"],
    label_smoothing_factor=TRAIN_CONFIG["label_smoothing_factor"],
    fp16=(DEVICE == "cuda"),
    save_strategy="steps",
    save_steps=TRAIN_CONFIG["save_steps"],
    save_total_limit=TRAIN_CONFIG["save_total_limit"],
    evaluation_strategy="steps",
    eval_steps=TRAIN_CONFIG["eval_steps"],
    predict_with_generate=True,
    generation_num_beams=TRAIN_CONFIG["eval_num_beams"],
    generation_max_length=TRAIN_CONFIG["eval_generation_max_length"],
    logging_steps=TRAIN_CONFIG["logging_steps"],
    report_to="none",
    seed=TRAIN_CONFIG["seed"],
    load_best_model_at_end=True,
    metric_for_best_model="chrf",
    greater_is_better=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding="longest", label_pad_token_id=-100),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2), SaveCurvesCallback()],
)

trainer.train()
merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print("Saved phase 2 model to", FINAL_DIR)


In [ ]:
# Cell 6 - Compare Phase 1 and Phase 2 model outputs on three examples
def translate_batch(model_ref: str, inputs: list[str]):
    local_model = AutoModelForSeq2SeqLM.from_pretrained(model_ref).to(DEVICE)
    local_model.eval()
    outputs = []
    for source in inputs:
        tokenizer.src_lang = "eng_Latn" if source.startswith("<en2ar>") else "arb_Arab"
        target_lang = "arb_Arab" if source.startswith("<en2ar>") else "eng_Latn"
        encoded = tokenizer(source, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
        generated = local_model.generate(
            **encoded,
            max_length=TRAIN_CONFIG["final_generation_max_length"],
            num_beams=TRAIN_CONFIG["final_num_beams"],
            forced_bos_token_id=tokenizer.lang_code_to_id[target_lang],
        )
        outputs.append(tokenizer.batch_decode(generated, skip_special_tokens=True)[0])
    return outputs

sampled = dataset_dict["validation"].shuffle(seed=TRAIN_CONFIG["seed"]).select(range(min(3, len(dataset_dict["validation"]))))
inputs = list(sampled["source"])
references = list(sampled["target"])
phase1_outputs = translate_batch(phase1_model_ref, inputs)
phase2_outputs = translate_batch(str(FINAL_DIR), inputs)
comparison_df = pd.DataFrame(
    {
        "source": inputs,
        "reference": references,
        "phase1_output": phase1_outputs,
        "phase2_output": phase2_outputs,
    }
)
display(comparison_df)


In [ ]:
# Cell 7 - Sweep checkpoints on 100 examples
def find_checkpoints(path: Path):
    return sorted(path.glob("checkpoint-*"), key=lambda item: int(item.name.split("-")[-1]))


def load_checkpoint_model(checkpoint_path: Path):
    eval_base = AutoModelForSeq2SeqLM.from_pretrained(phase1_model_ref)
    eval_model = PeftModel.from_pretrained(eval_base, checkpoint_path)
    eval_model = eval_model.merge_and_unload().to(DEVICE)
    eval_model.eval()
    return eval_model


def score_dataset(model_obj, dataset_split):
    sampled = dataset_split.shuffle(seed=TRAIN_CONFIG["seed"]).select(range(min(100, len(dataset_split))))
    predictions = []
    references = []
    for source, target in zip(sampled["source"], sampled["target"], strict=True):
        tokenizer.src_lang = "eng_Latn" if source.startswith("<en2ar>") else "arb_Arab"
        target_lang = "arb_Arab" if source.startswith("<en2ar>") else "eng_Latn"
        encoded = tokenizer(source, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
        generated = model_obj.generate(
            **encoded,
            max_length=TRAIN_CONFIG["eval_generation_max_length"],
            num_beams=TRAIN_CONFIG["eval_num_beams"],
            forced_bos_token_id=tokenizer.lang_code_to_id[target_lang],
        )
        predictions.append(tokenizer.batch_decode(generated, skip_special_tokens=True)[0].strip())
        references.append(target.strip())
    return {
        "bleu": corpus_bleu(predictions, [references]).score,
        "chrf": chrf_metric.corpus_score(predictions, [references]).score,
    }

checkpoint_rows = []
for checkpoint in find_checkpoints(CKPT_DIR):
    checkpoint_model = load_checkpoint_model(checkpoint)
    train_scores = score_dataset(checkpoint_model, dataset_dict["train"])
    val_scores = score_dataset(checkpoint_model, dataset_dict["validation"])
    checkpoint_rows.append(
        {
            "checkpoint": checkpoint.name,
            "train_bleu": train_scores["bleu"],
            "train_chrf": train_scores["chrf"],
            "val_bleu": val_scores["bleu"],
            "val_chrf": val_scores["chrf"],
        }
    )

checkpoint_df = pd.DataFrame(checkpoint_rows)
display(checkpoint_df)


In [ ]:
# Cell 8 - Plot checkpoint metrics
if "checkpoint_df" in globals() and not checkpoint_df.empty:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    checkpoint_df.plot(x="checkpoint", y=["train_chrf", "val_chrf"], kind="bar", ax=axes[0], color=["#0f766e", "#c97324"])
    axes[0].set_title("chrF++ by checkpoint")
    checkpoint_df.plot(x="checkpoint", y=["train_bleu", "val_bleu"], kind="bar", ax=axes[1], color=["#115e59", "#d97706"])
    axes[1].set_title("BLEU by checkpoint")
    plt.tight_layout()
    plt.show()
else:
    print("No checkpoint metrics available yet.")


In [ ]:
# Cell 9 - Training curves
curve_rows = json.loads(CURVES_PATH.read_text(encoding="utf-8")) if CURVES_PATH.exists() else []
curves = pd.DataFrame(curve_rows)
if not curves.empty:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    if "loss" in curves:
        curves.dropna(subset=["loss"]).plot(x="step", y="loss", ax=axes[0], color="#0f766e", label="train_loss")
    if "eval_loss" in curves:
        curves.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=axes[0], color="#c97324", label="val_loss")
    axes[0].set_title("Loss curves")
    metric_columns = [column for column in ["eval_chrf", "eval_bleu"] if column in curves.columns]
    if metric_columns:
        curves.dropna(subset=metric_columns).plot(x="step", y=metric_columns, ax=axes[1], color=["#115e59", "#d97706"])
    axes[1].set_title("Validation metrics")
    plt.tight_layout()
    plt.show()
else:
    print("No curve data found yet.")


In [ ]:
# Cell 10 - Final comparison table
if "comparison_df" in globals():
    display(comparison_df)
